# Deportation Data Project: ICE data

Data source: [Deportation Data Project](https://deportationdata.org/data/ice.html)

Our past story: [ICE arrests](https://coloradosun.com/2025/12/31/ice-arrests-2025-data-deportation-data-project/)

Related stories:

- [NYT interactive records](https://www.nytimes.com/interactive/2025/12/04/us/ice-arrests-criminal-records-data.html)
- [NYT interactive network](https://www.nytimes.com/interactive/2025/12/22/us/trump-immigration-deportation-network-ice-arrests.html)
- [CO Times Recorder about Aurora center](https://coloradotimesrecorder.com/2026/03/after-co-times-recorder-revealed-secret-detention-centers-in-co-27-lawmakers-call-on-ice-for-immediate-transparency/77113/)


In [2]:
import polars as pl  # polars is much faster than pandas for large data
from datetime import datetime
import matplotlib.pyplot as plt
from plotnine import ggplot, aes, geom_line, geom_bar, theme_minimal

from helpers import (
    read_data,
    read_calls_data,
    read_historical_data,
    clean_duplicates,
    confirm_state,
    state_from_landmark,
    state_from_docket,
    co_facilities,
    fbi_violent_crime,
    get_percent,
)

In [ ]:
# TODO: I think we should start open-sourcing our methodology/scripts
# TODO: would data reading be faster if we specified columns?

# Data

## Arrests

Description:

> Records every time ICE arrests someone, whether or not that arrest results in a decision to detain the person; or issues a Notice to Appear (NTA), the document that starts a deportation case. Note that other agencies also issue NTAs, and those would not appear as arrests in this table. We treat "apprehensions," "arrests," and "administrative arrests" as synonyms.


In [ ]:
arrests = read_data("arrests")

### Cleaning

1. Filter for blanks in the Apprehension State field. If Apprehension Landmark Site appears to be in Colorado, update the Apprehension State accordingly.
   1. Note: facility is probably unrelated to apprehension - people have been sent to random facilities in other states.
2. Clean out states listed in the Apprehension State field and remove those that don't belong.
3. Search for duplicates on unique identifiers. If they occurred within a day or two get rid of the older one (often there's removal on one of them, keep that one). TODO: but isn't it meaningful that someone is arrested more than once?
   1. Changed to use the "duplicate_likely" column provided by Deportation Data Project
4. If you plan to compare across years, consider filtering prior years to only the dates available in the current year. For comparisons across administrations, keep in mind that Inauguration Day is January 20.


Appears to be a data issue with impossible date of arrest


In [ ]:
arrests = arrests.with_columns(
    arrest_year=pl.col("apprehension_date_time").dt.year(),
    arrest_month=pl.col("apprehension_date_time").dt.month(),
    arrest_day=pl.col("apprehension_date_time").dt.day(),
)

display(arrests["arrest_year"].value_counts())
arrests["apprehension_date"].sort().value_counts()

In [ ]:
arrests = clean_duplicates(
    arrests, id_col="unique_identifier", datetime_col="apprehension_date"
)

In [ ]:
print(arrests["apprehension_date"].min(), arrests["apprehension_date"].max())

## Detention stays

Description:

> Records each detention from book-in to book-out at a given detention center for a given individual; most individuals have more than one row in the table because they are transferred between detention centers. Individuals may also be detained more than once and therefore have more than one "stay" in detention. We explain further in the relevant field definitions.


In [ ]:
stays = read_data(
    "detention-stays", schema_overrides={"initial_bond_set_date": pl.Date}
)

In [ ]:
# this gets a list of the detention facilities
det_facility = stays.group_by(["detention_facility_first"]).agg(pl.len().alias("count"))
det_facility.write_csv("output/det_facility.csv")
det_facility

In [ ]:
# this uses the list of colorado facilities to extract the info from the larger detention facility list
co_detain = stays.filter(pl.col("detention_facility_first").is_in(co_facilities))
co_detain.write_csv("output/co_detain.csv")
co_detain.head()

## Detention stints

Description:

> Individual book-ins to each detention facility (multiple per stay when detainees are transferred). We recommend using the detention stays dataset for most detention analyses at the individual level. For analyses at the detention facility level, by contrast, we recommend using the detention stints dataset. Note that filtering by stint can lead to inaccurate individual-level inferences, since a single person may appear across multiple stints when transferred between facilities.


In [ ]:
stints = read_data(
    "detention-stints",
    schema_overrides={
        "initial_bond_set_date": pl.Date,
        "initial_bond_set_amount": pl.String,
        "bond_posted_date": pl.String,
        "bond_posted_amount": pl.String,
        "final_order_date": pl.String,
        "likely_duplicate": pl.String,
    },
)

## 911 calls


In [ ]:
firecalls = read_calls_data("calls-afr")
pdcalls = read_calls_data("calls-apd")

firecalls


## Historical data

We did not end up using these data because they are too difficult to clean and understand confidently.


In [ ]:
# hd1 = read_historical_data(
#     "ERO Apprehension",
#     header_row=4,
#     schema_overrides={"Fugitive as of Date": pl.Datetime, "Departed Date": pl.Datetime},
# )

In [ ]:
# h2 = read_historical_data(
#     "Redacted",
#     header_row=6,
#     drop_cols=[
#         "ARREST MOST SERIOUS CONVICTION DATE_1",
#         "EID PERS ID",
#         "ARREST MOST SERIOUS CONVICTION CHARGE DATE_1",
#         "ARREST MOST SERIOUS PENDING CRIMINAL CHARGE CATEGORY",
#     ],
# )

In [ ]:
# all years, but no state info - cannot use
mapping = {"Arrested Created By": "Arrest Created By"}

# h3 = read_data(
#     "Redacted_raw.xlsx",
#     header_row=5,
#     rename_map=mapping,
# )

In [ ]:
# hd4 = read_data(
#     "UN",
#     schema_overrides={"Departed Date": pl.Datetime},
#     header_row=5,
#     rename_map=mapping,
#     drop_cols=["Removal Threat Level", "Case Closed Date"],
# )

We can concatenate the datasets because the end of the historical dataset exactly coincides with the start of the next.


In [ ]:
# print(hd1["Apprehension Date And Time"].min(), hd1["Apprehension Date And Time"].max())

### Cleaning

Remove duplicates if within 24 hours of each other

TODO: look at their code for detecting duplicates


In [ ]:
# # I don't want to drop a third of the dataset, but it's hard to tell which are duplicates...
# hd1 = clean_duplicates(hd1)

# # ensure no overlap with next set
# hd1 = hd1.filter(pl.col("apprehension_date") < pl.date(2022, 10, 1))

In [ ]:
# no id col provided for hd2

In [ ]:
# hd3 = clean_duplicates(
#     hd3, id_col="Anonymized Identifier", datetime_col="Apprehension Date"
# )

In [ ]:
# no id col provided for hd4

## Construct full dataset


In [ ]:
# most accurate information is most recent
# keep one row per unique person and their MSC
charge = (
    stays.filter(pl.col("msc_charge").is_not_null())
    .sort("stay_book_out_date_time")
    .unique(subset=["unique_identifier"], keep="last")[
        ["unique_identifier", "msc_charge"]
    ]
)

In [ ]:
# arrests with Denver area of responsibility AND state of Colorado
co_arrests = arrests.filter(confirm_state(arrests) | state_from_landmark(arrests))

print(co_arrests.shape)
display(co_arrests.group_by("arrest_year").agg(pl.len()).sort("arrest_year"))
co_arrests.head(3)

In [ ]:
# filter to dates of interest
# not using 2023 data because of the large number of missing values for apprehension state
co_all = co_arrests.filter(pl.col("apprehension_date") >= pl.date(2024, 1, 1)).join(
    charge, on="unique_identifier", how="left"
)[
    "unique_identifier",
    "apprehension_date",
    "apprehension_criminality",
    "apprehension_method",
    "arrest_day",
    "arrest_month",
    "arrest_year",
    "msc_charge",
]

print(co_all.shape)
co_all.write_csv("output/co_all.csv")

In [ ]:
# # exploratory - not used
# co_hist = hd1.with_columns(
#     apprehension_state=pl.when(
#         state_from_landmark(
#             hd1,
#             aor_col="Apprehension AOR",
#             landmark_col="Apprehension Landmark"
#         ) | state_from_docket(hd1, toa_col="Physical Site",)
#     )
#     .then(pl.lit("COLORADO"))
#     .otherwise(pl.col("State"))
# ).filter(pl.col("apprehension_state") == "COLORADO")

# print("Records in CO", co_hist.shape)
# co_hist.write_csv("output/co_hist.csv")
# co_hist.group_by("arrest_year").agg(pl.len())

# co_hist = co_hist.with_columns(
#     unique_identifier=pl.col("Sequence Number/Unique Identifier").cast(pl.String),
#     msc_charge=pl.col("Most Serious Criminal Conviction"),
#     apprehension_criminality=pl.when(
#         pl.col("Most Serious Criminal Conviction").is_not_null()
#     )
#     .then(pl.lit("1 Convicted Criminal"))
#     .when(pl.col("Most Serious Criminal Charge").is_not_null())
#     .then(pl.lit("2 Pending Criminal Charges"))
#     .otherwise(pl.lit("3 Other Immigration Violator")),
#     apprehension_method=pl.col("Apprehension Method"),
# )
# print(co_hist["apprehension_criminality"].value_counts())
# print(co_arrests["apprehension_criminality"].value_counts())


# Analysis

Possible additional analyses: gender, age; citizenship, third country deportations
Fish's idea about which days there weren't any arrests, pulling out holidays


In [ ]:
co_all = pl.read_csv("output/co_all.csv").with_columns(
    apprehension_date=pl.col("apprehension_date").str.strptime(pl.Date, "%Y-%m-%d")
)

## Number of arrests

Running average to make trend more coherent.


For comparing to prior year, ensure that data for prior year up to current month is kept


In [ ]:
by_date = (
    co_all["apprehension_date"]
    .value_counts()
    .sort("apprehension_date")
    .with_columns(average=pl.col("count").fill_null(0).rolling_mean(window_size=7))[
        "average", "apprehension_date"
    ]
)

by_date.write_csv(f"output/by_date_{datetime.today().year}.csv")
by_date.sort("apprehension_date")["apprehension_date"].to_list()

plt.plot(by_date["apprehension_date"], by_date["average"])
# TODO: affected by CBP numbers so should take that out somehow, but CBP data not published yet

In [ ]:
# get the first two years of the past administrations
by_admin = co_all.filter(
    co_all["arrest_year"].is_in([2017, 2018, 2021, 2022, 2025, 2026])
    & (
        # odd years are complete except for inauguration day
        (co_all["arrest_year"] % 2 == 1)
        & ~((co_all["arrest_month"] == 1) & (co_all["arrest_day"] < 20))
        | (
            # even years are only complete through the max month/day available in current dataset
            (co_all["arrest_year"] % 2 == 0)
            & (co_all["arrest_month"] == co_all["apprehension_date"].max().month)
            & (co_all["arrest_day"] <= co_all["apprehension_date"].max().day)
        )
    )
).with_columns(
    admin_month=pl.col("arrest_month").cast(pl.Int32)
    + (12 * (1 - pl.col("arrest_year").cast(pl.Int32) % 2))
)

print(
    by_admin[["arrest_year", "apprehension_date", "admin_month"]]
    .sort("admin_month")
    .unique("admin_month")
)

# recode
by_admin = by_admin.with_columns(
    admin=pl.when(pl.col("arrest_year").is_in([2017, 2018]))
    .then(pl.lit("First Trump"))
    .when(pl.col("arrest_year").is_in([2021, 2022]))
    .then(pl.lit("Biden"))
    .otherwise(pl.lit("Second Trump"))
)

# each record is an arrest for a unique person
by_admin = by_admin.group_by(["admin", "admin_month"]).agg(pl.len())
by_admin = by_admin.sort(["admin", "admin_month"]).with_columns(
    cumsum=pl.col("len").fill_null(0).cum_sum().over("admin")
)

by_admin = by_admin.pivot(index=["admin_month"], columns="admin", values="cumsum")

by_admin.write_csv(f"output/by_admin_{datetime.today().year}.csv")
by_admin.sort("admin_month")

In [ ]:
totals = co_all.group_by("arrest_year").agg(pl.len())
totals.write_csv(f"output/totals_{datetime.today().year}.csv")
totals.sort("arrest_year")

## Criminality

No date restrictions applied because this is a percentage analysis, which is comparable across years.


In [ ]:
crim = co_all.with_columns(
    apprehension_criminality=pl.col("apprehension_criminality").str.replace(r"\d ", "")
)

crim = get_percent(crim, "arrest_year", "apprehension_criminality")

crim.write_csv(f"output/by_criminality_{datetime.today().year}.csv")
crim

## MSC (most serious charge)

Bar chart with percentages. For this one, I am not excluding dates since percentages are comparable.

- find list of top 5 crimes since Trump's inauguration


In [ ]:
# adds most serious conviction information, if present, to the arrests table
arrests_with_charge = co_all.filter(
    co_all["apprehension_criminality"].str.contains("Convicted")
)[
    [
        "unique_identifier",
        "msc_charge",
        "apprehension_date",
        "arrest_year",
        "arrest_month",
        "arrest_day",
    ]
]
print(arrests_with_charge.shape)
print("Top charges:")
display(
    arrests_with_charge.filter(pl.col("apprehension_date") > pl.date(2025, 1, 20))[
        "msc_charge"
    ]
    .value_counts()
    .sort("count", descending=True)
    .head(5)
)

In [ ]:
arrests_with_charge = arrests_with_charge.with_columns(
    violent=pl.when(pl.col("msc_charge").is_in(fbi_violent_crime).fill_null(False))
    .then(pl.lit("Violent"))
    .otherwise(pl.lit("Non-violent"))
)

arrests_with_charge = get_percent(arrests_with_charge, "arrest_year", "violent")

arrests_with_charge.write_csv("output/arrests_charges.csv")
arrests_with_charge.sort("arrest_year")

In [ ]:
# 17 of the people with no charge have an MSC listed - this is likely charged but not convicted
# TODO: see if the team generated the msc_charge column and if so how
charge.join(
    co_all.filter(pl.col("apprehension_criminality").str.contains("Other")),
    on="unique_identifier",
    how="inner",
)[["apprehension_criminality", "apprehension_date", "msc_charge"]]

## Length of detention stays over time

Due 20, due to run Memorial weekend
Aurora Detention Center operated by Geogroup

- shadowed people who were recently released
- DDP stays in "DENVER CONTRACT DETENTION FACILITY" check to see if aurora

Deliverables:

- rq1 + chart: length of stay over time, as far back as we can go
- rq2 + chart: anecdotally, fewer people are being released than are entering the facility?? monthly people in and out - is it possible to see if people were released
- colorado times recorder - able to see how long people were there - did they use a different data set

Questions:

- When did this facility open?


In [ ]:
aurora = (
    stints.filter(pl.col("detention_facility") == "DENVER CONTRACT DETENTION FACILITY")
    .filter(
        pl.col("stay_book_in_date_time") > pl.date(2022, 1, 1)
    )  # only 15 people were booked in 2022
    .with_columns(
        stint_len=pl.col("stay_book_out_date_time") - pl.col("stay_book_in_date_time"),
        stay_book_in_year=pl.col("stay_book_in_date_time").dt.year(),
        stay_book_in_month=pl.col("stay_book_in_date_time").dt.month(),
        stay_book_in_date=pl.col("stay_book_in_date_time").dt.date(),
        stay_book_out_date=pl.col("stay_book_out_date_time").dt.date(),
    )
    .with_columns(
        stint_days=(pl.col("stint_len") / pl.duration(days=1)).round(),
        stay_book_in_year_month=pl.col("stay_book_in_year").cast(pl.String)
        + "-"
        + pl.col("stay_book_in_month").cast(pl.String).str.zfill(2),
    )
    .sort("stint_len", descending=False)
    # remove duplicates
    .unique("unique_identifier", keep="first")
)

aurora["stay_book_in_year"].value_counts().sort("stay_book_in_year")

In [ ]:
# check that data is ordered
aurora.filter(pl.col("stay_book_out_date_time") < pl.col("stay_book_in_date_time"))

In [ ]:
# 6 people stayed shorter than an hour
aurora.filter(pl.col("stint_len") < pl.duration(hours=1))

In [ ]:
plt.plot(
    aurora["stint_days"].value_counts().sort("stint_days")["stint_days"],
    aurora["stint_days"].value_counts().sort("stint_days")["count"],
)

In [ ]:
plt.plot(
    aurora.sort("stay_book_in_date_time")["stay_book_in_date_time"],
    aurora.sort("stay_book_in_date_time")["stint_days"],
)

In [ ]:
by_date = aurora.with_columns(
    average=pl.col("stint_days").fill_null(0).rolling_mean(window_size=7)
).sort("stay_book_in_date")["average", "stay_book_in_date"]
plt.plot(by_date["stay_book_in_date"], by_date["average"])

In [ ]:
# number of people checking in per day
checkins = (
    aurora.group_by("stay_book_in_date")
    .agg(count=pl.len().cast(pl.Int64))
    .sort("stay_book_in_date")
)
plt.plot(checkins["stay_book_in_date"], checkins["count"])

In [ ]:
# number of people checking out per day
checkouts = (
    aurora.group_by("stay_book_out_date")
    .agg(count=pl.len().cast(pl.Int64))
    .sort("stay_book_out_date")
).with_columns()
plt.plot(checkouts["stay_book_out_date"], checkouts["count"])

In [ ]:
by_date

In [ ]:
stints["detention_release_reason"].value_counts().sort("count", descending=True)

In [ ]:
# net of people checking in per day

net_checkins = checkins.join(
    checkouts, left_on="stay_book_in_date", right_on="stay_book_out_date"
).with_columns(
    net=pl.col("count") - pl.col("count_right")  # net checkins minus checkouts per day
)

net_checkins = net_checkins.with_columns(
    stay_book_in_month=pl.col("stay_book_in_date").dt.year().cast(pl.String)
    + "-"
    + pl.col("stay_book_in_date").dt.month().cast(pl.String).str.zfill(2)
).sort("stay_book_in_date")

rolling = net_checkins.with_columns(average=pl.col("net").rolling_mean(window_size=7))[
    "average", "stay_book_in_date"
]

# plt.plot(rolling["stay_book_in_date"], rolling["average"])

total = (
    net_checkins.group_by("stay_book_in_month")
    .agg(total=pl.sum("net"))
    .sort("stay_book_in_month")
)
plt.bar(
    total["stay_book_in_month"],
    total["total"],
)

In [ ]:
# net of people checking in per day, for only nontransfers

# number of people checking out per day
checkouts2 = (
    aurora.filter(pl.col("detention_release_reason") != "Transferred")
    .group_by("stay_book_out_date")
    .agg(count=pl.len().cast(pl.Int64))
    .sort("stay_book_out_date")
)

net_checkins = checkins.join(
    checkouts2, left_on="stay_book_in_date", right_on="stay_book_out_date"
).with_columns(
    net=pl.col("count") - pl.col("count_right")  # net checkins minus checkouts per day
)

net_checkins2 = (
    net_checkins.with_columns(
        stay_book_in_month=pl.col("stay_book_in_date").dt.year().cast(pl.String)
        + "-"
        + pl.col("stay_book_in_date").dt.month().cast(pl.String).str.zfill(2)
    )
    .filter(pl.col("stay_book_in_month").is_not_null())
    .group_by("stay_book_in_month")
    .agg(pl.col("net").sum())
    .sort("stay_book_in_month")
)

plt.bar(net_checkins2["stay_book_in_month"], net_checkins2["net"])
net_checkins2

In [ ]:
by_month = (
    aurora.group_by("stay_book_in_year_month")
    .agg(pl.col("stint_days").mean())
    .sort("stay_book_in_year_month")
)
plt.bar(by_month["stay_book_in_year_month"], by_month["stint_days"])

In [ ]:
stays.filter(pl.col("detention_facility_first") == "DENVER CONTRACT DETENTION FACILITY")

## 911 calls

No due date, ASAP

- 911 calls for fire and police from Aurora facility
  - priority: which calls to find out more about
- WaPo use of force matching
- have the calls increased over time and what are the patterns for types of calls
- ID is call number
  - remove duplicate rows - printer caused


In [ ]:
by_date = (
    firecalls.with_columns(
        month=pl.col("DATE").dt.month().cast(pl.String).str.zfill(2)
        + "-"
        + pl.col("DATE").dt.year().cast(pl.String)
    )
    .group_by(["CALLTYPE","month"])
    .agg(pl.len().alias("count"))
    .sort("month")
)
display(firecalls.with_columns(year=pl.col("DATE").dt.year())['year'].value_counts().sort("year"))
ggplot(by_date, aes(x="month", y="count", group="CALLTYPE", color="CALLTYPE")) + geom_line() + theme_minimal()


In [ ]:
by_date = (
    pdcalls.with_columns(
        month=pl.col("DATE").dt.month().cast(pl.String).str.zfill(2)
        + "-"
        + pl.col("DATE").dt.year().cast(pl.String)
    )
    .group_by(["month"])
    .agg(pl.len().alias("count"))
    .sort("month")
)
plt.bar(by_date["month"], by_date["count"])
pdcalls.with_columns(year=pl.col("DATE").dt.year())['year'].value_counts().sort("year")

In [ ]:
# net of people checking in per day

net_checkins = checkins.join(
    checkouts, left_on="stay_book_in_date", right_on="stay_book_out_date"
).with_columns(
    net=pl.col("count") - pl.col("count_right")  # net checkins minus checkouts per day
)

net_checkins = net_checkins.with_columns(
    stay_book_in_month=pl.col("stay_book_in_date").dt.year().cast(pl.String)
    + "-"
    + pl.col("stay_book_in_date").dt.month().cast(pl.String).str.zfill(2)
).sort("stay_book_in_date")

rolling = net_checkins.with_columns(average=pl.col("net").rolling_mean(window_size=7))[
    "average", "stay_book_in_date"
]

# plt.plot(rolling["stay_book_in_date"], rolling["average"])

total = (
    net_checkins.group_by("stay_book_in_month")
    .agg(total=pl.sum("net"))
    .sort("stay_book_in_month")
)
plt.bar(
    total["stay_book_in_month"],
    total["total"],
)

In [ ]:
# net of people checking in per day

net_checkins = checkins.join(
    checkouts, left_on="stay_book_in_date", right_on="stay_book_out_date"
).with_columns(
    net=pl.col("count") - pl.col("count_right")  # net checkins minus checkouts per day
)

net_checkins = net_checkins.with_columns(
    stay_book_in_month=pl.col("stay_book_in_date").dt.year().cast(pl.String)
    + "-"
    + pl.col("stay_book_in_date").dt.month().cast(pl.String).str.zfill(2)
).sort("stay_book_in_date")

rolling = net_checkins.with_columns(average=pl.col("net").rolling_mean(window_size=7))[
    "average", "stay_book_in_date"
]

# plt.plot(rolling["stay_book_in_date"], rolling["average"])

total = (
    net_checkins.group_by("stay_book_in_month")
    .agg(total=pl.sum("net"))
    .sort("stay_book_in_month")
)
plt.bar(
    total["stay_book_in_month"],
    total["total"],
)

In [ ]:
firecalls["CALLTYPE"].value_counts().sort("count", descending=True)

In [ ]:
pdcalls["CALLTYPE"].value_counts().sort("count", descending=True)

# Unused datasets (holdover from last story)

## Detainers

From the description: "Records all requests to state, county, and municipal jails and prisons either for a person to be held on a detainer or for a notification of release date and time. A detainer is a request to a local jail to hold someone for 48 hours beyond when they otherwise would be released so that ICE can make an arrest in the jail while the individual remains detained."


In [ ]:
# detainer = read_data("detainer")

In [ ]:
# felon_lookup = detainer[
#     [
#         "Unique Identifier",
#         "MSC Conviction Date",
#         "Detention Facility",
#         "Facility State",
#         "Prior Felony Yes No",
#         "Violent Misdemeanor Yes No",
#         "Aggravated Felony Yes No",
#     ]
# ].drop_duplicates(subset="Unique Identifier")
# felon_lookup.head()

In [ ]:
# # this adds more information about conviction date, facility and more; mostly used the conviction date
# arrests_supplement = arrests_with_charge.merge(
#     felon_lookup, on="Unique Identifier", how="left"
# )
# arrests_supplement.write_csv("output/arrests_plus.csv")
# arrests_supplement.head()


In [ ]:
# detainer.group_by(["Facility AOR"]).size().reset_index(name="counts")

In [ ]:
# detainer.group_by(["Facility State"]).size().reset_index(name="counts")

In [ ]:
# co_detainer = detainer[
#     (detainer["Facility AOR"] == "Denver Area of Responsibility")
#     | (detainer["Facility State"] == "Colorado")
#     # | (detainer["Facility State"] == "Wyoming")
# ]
# co_detainer.head()
# co_detainer.write_csv("output/co_wyo_detainer.csv")

## Removals

From the description: "Records every deportation that ICE conducts, with a row for each individual deportation. An individual only has more than one row if that individual was deported more than once. Note that expulsions may occur directly at the border, by CBP, without involving ICE."


In [ ]:
# removals = read_data("removals")

In [ ]:
# co__wyo_remove = removals[
#     (removals["Docket AOR"] == "Denver Area of Responsibility")
#     | (removals["Apprehension State"] == "Colorado")
#     | (removals["Apprehension State"] == "Wyoming")
# ]

## Encounters

From the description: "Records every time ICE Enforcement and Removal Operations encounters a person, i.e. considers whether to take enforcement action against a person. This need not mean a physical encounter. Most notably, every time ICE processes a match between FBI book-in information (i.e. to a jail or prison) and ICE database information, that match is logged as an ICE encounter. Generally, if an individual appears in the detainers or arrests table, that individual should appear in this table. An individual might appear in the removals or stays tables without appearing in the encounters data if Customs and Border Protection initially encounters the person. This is both the largest and the sparsest of the tables, and in many cases, encounters lack a unique ID because the individual lacked an A number (A numbers are generally only given to people with immigrant visas or when they are processed for deportation proceedings)."


In [ ]:
# encounters = read_data("encounters")

In [ ]:
# co_encounter = encounters[
#     encounters["Responsible AOR"] == "Denver Area of Responsibility"
# ]
# co_encounter.head()